In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/rainfall.csv", skiprows=8, skipinitialspace=True)

df["time"] = pd.to_datetime(df["time"])
df["rainfall_mm"] = (
    df["mean_GPCPMON_3_3_satellite_precip"]
    * df["time"].dt.days_in_month
)

df = df.sort_values("time").reset_index(drop=True)

print(df.head())
print(df.shape)


        time  mean_GPCPMON_3_3_satellite_precip  rainfall_mm
0 2000-01-01                          10.524809   326.269076
1 2000-02-01                          10.142104   294.121019
2 2000-03-01                           6.925753   214.698347
3 2000-04-01                           7.252641   217.579222
4 2000-05-01                           4.968721   154.030348
(312, 3)


In [2]:

df["rain_lag1"] = df["rainfall_mm"].shift(1)
df["rain_lag2"] = df["rainfall_mm"].shift(2)
df["rain_lag3"] = df["rainfall_mm"].shift(3)
df["rain_lag12"] = df["rainfall_mm"].shift(12)
df["rain_rolling3"] = df["rainfall_mm"].shift(1).rolling(3).mean()
df["month"] = df["time"].dt.month

df = df.dropna().reset_index(drop=True)

features = [
    "rain_lag1",
    "rain_lag2",
    "rain_lag3",
    "rain_lag12",
    "rain_rolling3",
    "month"
]

target = "rainfall_mm"

print(df[["time"] + features + [target]].head())
print("Total observations:", len(df))


        time   rain_lag1   rain_lag2   rain_lag3  rain_lag12  rain_rolling3  \
0 2001-01-01  244.023404  317.156409  345.146811  326.269076     302.108875   
1 2001-02-01  378.111331  244.023404  317.156409  294.121019     313.097048   
2 2001-03-01  279.401295  378.111331  244.023404  214.698347     300.512010   
3 2001-04-01  256.199726  279.401295  378.111331  217.579222     304.570784   
4 2001-05-01  303.243798  256.199726  279.401295  154.030348     279.614940   

   month  rainfall_mm  
0      1   378.111331  
1      2   279.401295  
2      3   256.199726  
3      4   303.243798  
4      5   194.594168  
Total observations: 300


In [3]:

train = df[df["time"].dt.year <= 2020].copy()
test = df[df["time"].dt.year >= 2021].copy()

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Training period:", train["time"].min(), "to", train["time"].max())
print("Testing period:", test["time"].min(), "to", test["time"].max())

Training observations: 240
Testing observations: 60
Training period: 2001-01-01 00:00:00 to 2020-12-01 00:00:00
Testing period: 2021-01-01 00:00:00 to 2025-12-01 00:00:00


In [4]:
assert train["time"].max() < test["time"].min()
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print("Chronological split verified!")


Chronological split verified!


In [5]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Rainfall model training completed!")
print("Training R²:", rf_model.score(X_train, y_train))
print("Testing R²:", rf_model.score(X_test, y_test))


Rainfall model training completed!
Training R²: 0.8145790464223663
Testing R²: 0.14738675724367756


In [6]:
y_pred = rf_model.predict(X_test)

results = test[["time", "rainfall_mm"]].copy()
results["predicted_rainfall"] = y_pred

print(results.head(10))


          time  rainfall_mm  predicted_rainfall
240 2021-01-01   392.949723          294.446412
241 2021-02-01   172.978939          236.737625
242 2021-03-01   262.137666          233.259074
243 2021-04-01   167.399769          251.817993
244 2021-05-01   304.366835          250.641784
245 2021-06-01   222.718663          250.250032
246 2021-07-01   224.493430          208.629466
247 2021-08-01   388.089357          233.110978
248 2021-09-01   331.177512          270.243752
249 2021-10-01   329.376035          347.776655


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf_mae = mean_absolute_error(y_test, y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rf_r2 = r2_score(y_test, y_pred)

baseline_previous = X_test["rain_lag1"]
baseline_seasonal = X_test["rain_lag12"]

comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Previous Month",
        "Same Month Last Year"
    ],
    "MAE": [
        rf_mae,
        mean_absolute_error(y_test, baseline_previous),
        mean_absolute_error(y_test, baseline_seasonal)
    ],
    "RMSE": [
        rf_rmse,
        np.sqrt(mean_squared_error(y_test, baseline_previous)),
        np.sqrt(mean_squared_error(y_test, baseline_seasonal))
    ],
    "R²": [
        rf_r2,
        r2_score(y_test, baseline_previous),
        r2_score(y_test, baseline_seasonal)
    ]
})

print(comparison.to_string(index=False))


               Model       MAE      RMSE        R²
       Random Forest 53.989835 67.245976  0.147387
      Previous Month 75.573868 94.864966 -0.696802
Same Month Last Year 63.472366 78.811238 -0.171105


In [8]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

joblib.dump(
    {"model": rf_model, "features": features},
    "../models/rainfall_rf.joblib"
)

print("Rainfall model saved successfully!")


Rainfall model saved successfully!
